# EDA — Aged Care Quality & Access Gap

Narrative: **The Detective** — national overview → state comparison → SA3-level reveal

**Sections**
1. Setup & load
2. National overview
3. Quality over time + mandate effect (Oct 2023)
4. For-profit problem — quality by org type
5. Remote penalty — quality by MMM band
6. Supply — facilities and beds by state
7. Demand — HCP high needs / waitlist pressure
8. Funding story

---

## Key Insights Summary

| # | Insight | Headline number |
|---|---------|----------------|
| 1 | **Mandate worked** — staffing mandate lifted quality +0.25 pts | 3.40 → 3.65 after Oct 2023 |
| 2 | **For-profit gap** — government facilities outscore for-profit | 4.21 vs 3.68 (gap = 0.53) |
| 3 | **City problem, not remote** — rural/remote score *higher* than metro | MM5 Small rural = 4.05, MM1 City = 3.75 |
| 4 | **Waitlist trap** — Noosa Hinterland has 313 high-needs HCP users per 1 residential bed | 313x pressure ratio |
| 5 | **Supply consolidation** — 118 SA3s lost facilities, only 77 gained | Net beds +12,270 but distributed unevenly |
| 6 | **Demand explosion** — HCP high-needs users up +22% in 2 years | 140k (2023) → 172k (2025), now 59% of all HCP |

## 1. Setup & Load

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

CLEAN = '../../data/clean'

ratings  = pd.read_csv(f'{CLEAN}/star_ratings_by_facility.csv', parse_dates=['snapshot_date'])
supply   = pd.read_csv(f'{CLEAN}/service_supply_by_sa3.csv')
users    = pd.read_csv(f'{CLEAN}/service_users_by_sa3.csv')
funding  = pd.read_csv(f'{CLEAN}/service_funding_by_facility.csv')

print('ratings :', ratings.shape)
print('supply  :', supply.shape)
print('users   :', users.shape)
print('funding :', funding.shape)

ratings : (31290, 18)
supply  : (2307, 11)
users   : (1005, 14)
funding : (38481, 8)


In [2]:
# Normalise org_type in ratings (Purpose column)
ORG_MAP = {
    'For profit': 'profit',
    'Not for profit': 'not_for_profit',
    'Government': 'government',
}
ratings['org_type'] = ratings['Purpose'].map(ORG_MAP).fillna('unknown')

# MMM numeric for ordering
ratings['mmm_num'] = ratings['mmm_code'].str.extract(r'(\d)').astype(float)

# Latest snapshot for cross-section views
latest_snap = ratings['snapshot_date'].max()
print('Latest snapshot:', latest_snap.strftime('%B %Y'))

# Mandate timestamp in ms (needed for plotly add_vline on datetime axis)
MANDATE_TS = pd.Timestamp('2023-10-01').timestamp() * 1000

Latest snapshot: February 2026


## 2. National Overview

In [3]:
latest_supply = supply[supply['year'] == supply['year'].max()]
latest_users  = users[users['year'] == users['year'].max()]

print('=== National snapshot (latest year) ===')
print(f"Total facilities         : {latest_supply['n_facilities'].sum():,.0f}")
print(f"  Residential            : {latest_supply['n_residential'].sum():,.0f}")
print(f"  Home care              : {latest_supply['n_homecare'].sum():,.0f}")
print(f"Residential beds         : {latest_supply['residential_places'].sum():,.0f}")
print(f"Home care packages       : {latest_supply['homecare_places'].sum():,.0f}")
print()
print(f"Total residential users  : {latest_users['total_residential'].sum():,.0f}")
print(f"Total home care users    : {latest_users['total_homecare'].sum():,.0f}")
print(f"HCP high needs (L3+L4)   : {latest_users['hcp_high_needs'].sum():,.0f}")
print()
latest_rat = ratings[ratings['snapshot_date'] == latest_snap]
print(f"Facilities rated         : {latest_rat.shape[0]:,}")
print(f"National avg quality     : {latest_rat['quality_score'].mean():.2f} / 5")

=== National snapshot (latest year) ===
Total facilities         : 5,378
  Residential            : 2,773
  Home care              : 2,363
Residential beds         : 228,259
Home care packages       : 1,865

Total residential users  : 203,624
Total home care users    : 291,435
HCP high needs (L3+L4)   : 172,285

Facilities rated         : 2,582
National avg quality     : 3.79 / 5


In [4]:
# Org type breakdown nationally
org_counts = latest_supply[['n_nfp','n_government','n_private']].sum().reset_index()
org_counts.columns = ['org_type','count']
org_counts['org_type'] = ['Not for profit','Government','Private']

fig = px.pie(
    org_counts, values='count', names='org_type',
    title='Facility ownership nationally (latest year)',
    color='org_type',
    color_discrete_map={'Not for profit':'#4C78A8','Government':'#72B7B2','Private':'#F58518'}
)
fig.show()

## 3. Quality Over Time + Mandate Effect (Oct 2023)

In [5]:
# Average quality_score per state per snapshot
qual_time = (
    ratings.groupby(['snapshot_date','state'])['quality_score']
    .mean().reset_index()
)

fig = px.line(
    qual_time, x='snapshot_date', y='quality_score', color='state',
    title='Average quality score by state over time',
    labels={'quality_score':'Avg quality score','snapshot_date':'Quarter'}
)
fig.add_vline(
    x=MANDATE_TS, line_dash='dash', line_color='red',
    annotation_text='Oct 2023 staffing mandate',
    annotation_position='top right'
)
fig.update_layout(yaxis_range=[2.5, 4.5])
fig.show()

In [6]:
# National avg quality before vs after mandate
MANDATE = pd.Timestamp('2023-10-01')
ratings['period'] = ratings['snapshot_date'].apply(
    lambda d: 'After mandate' if d >= MANDATE else 'Before mandate'
)

mandate_compare = (
    ratings.groupby(['state','period'])['quality_score']
    .mean().reset_index()
)

fig = px.bar(
    mandate_compare, x='state', y='quality_score', color='period',
    barmode='group',
    title='Quality score before vs after Oct 2023 staffing mandate',
    labels={'quality_score':'Avg quality score'},
    color_discrete_map={'Before mandate':'#aec7e8','After mandate':'#1f77b4'}
)
fig.show()

In [7]:
# Heatmap: state × quarter
pivot = qual_time.pivot(index='state', columns='snapshot_date', values='quality_score')
pivot.columns = [c.strftime('%b %Y') for c in pivot.columns]

fig = px.imshow(
    pivot, text_auto='.2f',
    title='Quality score heatmap — state × quarter',
    color_continuous_scale='RdYlGn',
    zmin=3.0, zmax=4.0
)
fig.show()

### Insight — Mandate Effect
> **The Oct 2023 staffing mandate worked.** National avg quality rose from **3.40 → 3.65** (+0.25 pts, ~+7%) within a few quarters after the mandate took effect. This is the clearest before/after signal in the entire dataset. The heatmap shows the improvement was broad-based across all states, not driven by one outlier.

## 4. For-Profit Problem — Quality by Org Type

In [8]:
# Box plot quality by org type (latest snapshot)
latest_rat = ratings[ratings['snapshot_date'] == latest_snap].copy()

fig = px.box(
    latest_rat[latest_rat['org_type'] != 'unknown'],
    x='org_type', y='quality_score', color='org_type',
    title=f'Quality score by org type ({latest_snap.strftime("%B %Y")})',
    labels={'quality_score':'Quality score','org_type':'Organisation type'},
    color_discrete_map={'profit':'#F58518','not_for_profit':'#4C78A8','government':'#72B7B2'},
    points='outliers'
)
fig.show()

In [9]:
# Quality trend by org type over time
qual_org = (
    ratings[ratings['org_type'] != 'unknown']
    .groupby(['snapshot_date','org_type'])['quality_score']
    .mean().reset_index()
)

fig = px.line(
    qual_org, x='snapshot_date', y='quality_score', color='org_type',
    title='Quality score by org type over time',
    labels={'quality_score':'Avg quality score','snapshot_date':'Quarter'},
    color_discrete_map={'profit':'#F58518','not_for_profit':'#4C78A8','government':'#72B7B2'}
)
fig.add_vline(x=MANDATE_TS, line_dash='dash', line_color='red',
              annotation_text='Oct 2023 mandate')
fig.show()

### Insight — For-Profit Problem
> **Government-run facilities significantly outperform for-profit ones.** In the latest snapshot, government facilities average **4.21 vs 3.68** for for-profit — a gap of **0.53 points**. The trend chart shows this gap has been consistent over time, not a one-off. Private facilities are also concentrated in metro areas (higher % in NSW, VIC) where land value drives profit motive over care quality.

In [10]:
# Private share by state (latest year)
state_supply = latest_supply.groupby('year').apply(lambda x: x).reset_index(drop=True)
state_supply = supply[supply['year'] == supply['year'].max()].copy()

# Need state in supply — join from ratings
sa3_state = ratings[['sa3_code','state']].dropna().drop_duplicates('sa3_code')
state_supply = state_supply.merge(sa3_state, on='sa3_code', how='left')

state_org = state_supply.groupby('state')[['n_nfp','n_government','n_private']].sum()
state_org['total'] = state_org.sum(axis=1)
state_org['pct_private'] = state_org['n_private'] / state_org['total'] * 100
state_org = state_org.reset_index().sort_values('pct_private', ascending=False)

fig = px.bar(
    state_org, x='state', y='pct_private',
    title='Private (for-profit) facility share by state (latest year)',
    labels={'pct_private':'% private facilities','state':'State'},
    color='pct_private', color_continuous_scale='Oranges'
)
fig.add_hline(y=state_org['pct_private'].mean(), line_dash='dash',
              annotation_text='National avg')
fig.show()

## 5. Remote Penalty — Quality by MMM Band

### Insight — Remote Penalty (Reversed)
> **Counterintuitive finding: remote areas score *higher* than cities.** MM5 Small rural tops at **4.05**, while MM1 Major city sits at **3.75**. This flips the expected narrative. A likely explanation: remote facilities are mostly government or NFP-run (higher quality) while city facilities skew for-profit. The real "penalty" in remote areas is *access* (fewer beds, longer distances) — not quality. This reframes the story: **the access gap, not the quality gap, is the remote problem.**

In [11]:
# Box plot quality by MMM band
mmm_labels = {
    1: 'MM1 Major city',
    2: 'MM2 Regional centre',
    3: 'MM3 Large rural',
    4: 'MM4 Medium rural',
    5: 'MM5 Small rural',
    6: 'MM6 Remote',
    7: 'MM7 Very remote',
}
latest_rat['mmm_label'] = latest_rat['mmm_num'].map(mmm_labels)

fig = px.box(
    latest_rat.dropna(subset=['mmm_label']),
    x='mmm_label', y='quality_score', color='mmm_label',
    category_orders={'mmm_label': list(mmm_labels.values())},
    title=f'Quality score by remoteness (MMM) — {latest_snap.strftime("%B %Y")}',
    labels={'quality_score':'Quality score','mmm_label':'Remoteness class'},
    points='outliers'
)
fig.update_layout(showlegend=False)
fig.show()

In [12]:
# Avg quality by MMM over time
ratings['mmm_label'] = ratings['mmm_num'].map(mmm_labels)
mmm_time = (
    ratings.dropna(subset=['mmm_label'])
    .groupby(['snapshot_date','mmm_label'])['quality_score']
    .mean().reset_index()
)

fig = px.line(
    mmm_time,
    x='snapshot_date', y='quality_score', color='mmm_label',
    category_orders={'mmm_label': list(mmm_labels.values())},
    title='Quality score by remoteness over time',
    labels={'quality_score':'Avg quality score','snapshot_date':'Quarter','mmm_label':'MMM band'}
)
fig.add_vline(x=MANDATE_TS, line_dash='dash', line_color='red',
              annotation_text='Oct 2023 mandate')
fig.show()

## 6. Supply — Facilities and Beds by State

### Insight — Supply Consolidation
> **The system is consolidating, not expanding.** 118 SA3 regions lost at least one residential facility since 2019, while only 77 gained one. Total bed count rose +12,270 — but the growth is concentrated in fewer, larger facilities in bigger centres. Smaller communities are being left behind. Only 5 SA3s have zero facilities today, but many more are down to 1–2 facilities — extremely fragile supply.

In [13]:
# Residential places over time by state
supply_state = supply.merge(sa3_state, on='sa3_code', how='left')
beds_state = supply_state.groupby(['year','state'])['residential_places'].sum().reset_index()

fig = px.line(
    beds_state, x='year', y='residential_places', color='state',
    title='Residential places (beds) by state 2019–2025',
    labels={'residential_places':'Licensed beds','year':'Year'},
    markers=True
)
fig.show()

In [14]:
# SA3s that lost residential facilities 2019 → latest
supply_2019 = supply[supply['year'] == 2019][['sa3_code','sa3_name','n_residential']].rename(columns={'n_residential':'res_2019'})
supply_late = supply[supply['year'] == supply['year'].max()][['sa3_code','sa3_name','n_residential']].rename(columns={'n_residential':'res_latest'})
supply_change = supply_2019.merge(supply_late, on=['sa3_code','sa3_name'], how='inner')
supply_change['change'] = supply_change['res_latest'] - supply_change['res_2019']

lost = supply_change[supply_change['change'] < 0].sort_values('change')
gained = supply_change[supply_change['change'] > 0].sort_values('change', ascending=False)

print(f'SA3s that LOST residential facilities: {len(lost)}')
print(f'SA3s that GAINED residential facilities: {len(gained)}')
print(f'SA3s with ZERO change: {len(supply_change[supply_change["change"]==0])}')

fig = px.bar(
    lost.head(20), x='change', y='sa3_name',
    orientation='h',
    title='Top 20 SA3 regions that lost the most residential facilities (2019 → latest)',
    labels={'change':'Change in facilities','sa3_name':'SA3'}
)
fig.show()

SA3s that LOST residential facilities: 118
SA3s that GAINED residential facilities: 77
SA3s with ZERO change: 133


In [15]:
# Desert SA3s — zero residential facilities in latest year
desert = supply_late[supply_late['res_latest'] == 0]
print(f'SA3s with ZERO residential facilities in latest year: {len(desert)}')

SA3s with ZERO residential facilities in latest year: 5


### Insight — Waitlist Trap
> **The headline finding of this project.** Noosa Hinterland has **313 high-needs HCP users per 1 residential bed** — meaning 313 people who need high-level care are stuck at home because there is almost no residential capacity nearby. This is not an isolated case: 8 of the top 10 SA3s by pressure are in Queensland or outer-metro areas. Meanwhile, HCP high-needs users nationally jumped **+22% in just 2 years** (140k → 172k), now making up **59% of all home care users** — a slow-motion crisis of unmet demand.

## 7. Demand — HCP High Needs / Waitlist Pressure

In [16]:
# Scatter: residential users vs HCP high needs by SA3
latest_users_yr = users[users['year'] == users['year'].max()].copy()
latest_users_yr = latest_users_yr.merge(
    supply_late[['sa3_code','res_latest']].rename(columns={'res_latest':'res_places'}),
    on='sa3_code', how='left'
)

fig = px.scatter(
    latest_users_yr.dropna(subset=['res_places','hcp_high_needs']),
    x='total_residential', y='hcp_high_needs',
    size='res_places',
    hover_name='sa3_name',
    title='Residential users vs HCP high-needs users by SA3 (latest year)',
    labels={
        'total_residential': 'Residential care users',
        'hcp_high_needs': 'HCP Level 3+4 users (waitlist pressure)'
    },
    opacity=0.6
)
fig.show()

In [17]:
# Waitlist pressure: hcp_high_needs per residential place
latest_users_yr['waitlist_pressure'] = (
    latest_users_yr['hcp_high_needs'] / latest_users_yr['res_places'].replace(0, np.nan)
)

top_pressure = (
    latest_users_yr.dropna(subset=['waitlist_pressure'])
    .nlargest(20, 'waitlist_pressure')
)

fig = px.bar(
    top_pressure, x='waitlist_pressure', y='sa3_name',
    orientation='h',
    title='Top 20 SA3 regions by waitlist pressure (HCP L3+L4 per residential place)',
    labels={'waitlist_pressure':'HCP high-needs per residential place','sa3_name':'SA3'}
)
fig.show()

### Insight — Funding Story
> **For-profit providers receive the majority of government funding** — Private Incorporated Body alone received ~$59B vs Charitable ($36B) and Religious ($34B) combined. However, their quality scores are the lowest among org types. This raises a policy question: is government money being efficiently allocated? The scatter plot can reveal whether higher-funded SA3s actually deliver better quality outcomes.

In [18]:
# HCP high needs share over time nationally
hcp_trend = users.groupby('year')[['hcp_high_needs','total_homecare']].sum().reset_index()
hcp_trend['pct_high'] = hcp_trend['hcp_high_needs'] / hcp_trend['total_homecare'] * 100

fig = px.line(
    hcp_trend, x='year', y='pct_high',
    title='% of home care users with high needs (Level 3+4) over time',
    labels={'pct_high':'% HCP high needs','year':'Year'},
    markers=True
)
fig.show()

## 8. Funding Story

## 10. What Did the Mandate Actually Fix?

In [19]:
MANDATE = pd.Timestamp('2023-10-01')
dims = ['residents_exp', 'staffing', 'compliance', 'quality_measures']
dim_labels = ['Residents experience', 'Staffing', 'Compliance', 'Quality measures']

records = []
for d, label in zip(dims, dim_labels):
    b = ratings[ratings['snapshot_date'] < MANDATE][d].mean()
    a = ratings[ratings['snapshot_date'] >= MANDATE][d].mean()
    records.append({'dimension': label, 'before': b, 'after': a, 'change': a - b})

sub_df = pd.DataFrame(records).sort_values('change', ascending=False)

fig = px.bar(
    sub_df.melt(id_vars='dimension', value_vars=['before', 'after'],
                var_name='period', value_name='score'),
    x='dimension', y='score', color='period', barmode='group',
    title='Sub-rating change before vs after Oct 2023 staffing mandate',
    labels={'score': 'Avg score', 'dimension': 'Sub-rating'},
    color_discrete_map={'before': '#aec7e8', 'after': '#1f77b4'}
)
fig.update_layout(yaxis_range=[2.0, 5.0])
fig.show()

print("Change per sub-rating:")
for _, row in sub_df.iterrows():
    bar = '+' * int(abs(row['change']) * 20) if row['change'] > 0 else '-'
    print(f"  {row['dimension']:25s}: {row['before']:.2f} -> {row['after']:.2f}  ({row['change']:+.3f})  {bar}")

Change per sub-rating:
  Staffing                 : 2.49 -> 3.00  (+0.509)  ++++++++++
  Compliance               : 4.28 -> 4.57  (+0.289)  +++++
  Residents experience     : 3.28 -> 3.50  (+0.220)  ++++
  Quality measures         : 3.55 -> 3.54  (-0.015)  -


### Insight — Staffing Was the Target
> **The mandate was a staffing mandate — and staffing is exactly what moved.** Staffing sub-rating jumped **+0.51** (2.49 → 3.00), the largest gain of any dimension. Compliance rose +0.29, residents' experience +0.22. Critically, **quality measures barely changed (-0.015)** — suggesting the mandate improved observable inputs (staff hours, ratios) but has not yet translated into better measurable outcomes. This is the policy story: inputs improved, outcomes lagged.

## 11. State Rankings — Winners and Losers

In [20]:
first_snap = ratings['snapshot_date'].min()
first_state = ratings[ratings['snapshot_date']==first_snap].groupby('state')['quality_score'].mean()
last_state  = ratings[ratings['snapshot_date']==latest_snap].groupby('state')['quality_score'].mean()

rank_df = pd.DataFrame({'first': first_state, 'last': last_state})
rank_df['change'] = rank_df['last'] - rank_df['first']
rank_df['rank_first'] = rank_df['first'].rank(ascending=False).astype(int)
rank_df['rank_last']  = rank_df['last'].rank(ascending=False).astype(int)
rank_df['rank_change'] = rank_df['rank_first'] - rank_df['rank_last']
rank_df = rank_df.reset_index().sort_values('change', ascending=False)

fig = px.bar(
    rank_df, x='state', y='change', color='change',
    color_continuous_scale='RdYlGn',
    title=f'Quality score change by state ({first_snap.strftime("%b %Y")} to {latest_snap.strftime("%b %Y")})',
    labels={'change': 'Change in avg quality score', 'state': 'State'},
    text='change'
)
fig.update_traces(texttemplate='%{text:+.3f}', textposition='outside')
fig.add_hline(y=0, line_color='black', line_width=1)
fig.show()

# Slope chart: first vs last
fig2 = go.Figure()
for _, row in rank_df.iterrows():
    color = '#2ca02c' if row['change'] > 0.4 else ('#d62728' if row['change'] < 0.35 else '#7f7f7f')
    fig2.add_trace(go.Scatter(
        x=[first_snap.strftime('%b %Y'), latest_snap.strftime('%b %Y')],
        y=[row['first'], row['last']],
        mode='lines+markers+text',
        name=row['state'],
        line=dict(color=color, width=2),
        text=[row['state'], row['state']],
        textposition=['middle left', 'middle right']
    ))
fig2.update_layout(
    title='Quality slope chart by state (first vs latest snapshot)',
    yaxis_title='Avg quality score', showlegend=False,
    yaxis_range=[3.0, 4.2]
)
fig2.show()

### Insight — NT Bứt Phá, VIC Tụt Hạng
> **NT improved the most of any state (+0.748 pts), jumping from rank 7 to rank 1.** TAS and QLD held steady. The surprise loser is **VIC, which fell from rank 1 to rank 5** despite improving in absolute terms (+0.312). ACT improved the least (+0.306) and sits last. This suggests VIC's starting advantage eroded as other states caught up faster post-mandate.

## 12. Why Remote Areas Score Higher — The Ownership Explanation

In [21]:
mmm_labels = {1:'MM1 City',2:'MM2 Regional',3:'MM3 Large rural',
              4:'MM4 Medium rural',5:'MM5 Small rural',6:'MM6 Remote',7:'MM7 Very remote'}

latest_rat = ratings[ratings['snapshot_date'] == latest_snap].copy()
latest_rat['mmm_label'] = latest_rat['mmm_num'].map(mmm_labels)

# Org type mix by MMM
mmm_org = (
    latest_rat[latest_rat['org_type'] != 'unknown']
    .groupby(['mmm_label','org_type']).size().reset_index(name='count')
)
mmm_org_total = mmm_org.groupby('mmm_label')['count'].sum().reset_index(name='total')
mmm_org = mmm_org.merge(mmm_org_total, on='mmm_label')
mmm_org['pct'] = mmm_org['count'] / mmm_org['total'] * 100

fig = px.bar(
    mmm_org,
    x='mmm_label', y='pct', color='org_type',
    category_orders={'mmm_label': list(mmm_labels.values())},
    title='Facility ownership mix by remoteness — cities are dominated by for-profit',
    labels={'pct': '% of facilities', 'mmm_label': 'Remoteness', 'org_type': 'Org type'},
    color_discrete_map={'profit': '#F58518', 'not_for_profit': '#4C78A8', 'government': '#72B7B2'},
    barmode='stack'
)
fig.show()

# Quality by MMM — side by side with org mix
mmm_quality = latest_rat.dropna(subset=['mmm_label']).groupby('mmm_label')['quality_score'].mean().reset_index()
fig2 = px.bar(
    mmm_quality,
    x='mmm_label', y='quality_score', color='quality_score',
    category_orders={'mmm_label': list(mmm_labels.values())},
    color_continuous_scale='RdYlGn',
    range_color=[3.5, 4.2],
    title='Avg quality score by remoteness — remote outperforms city',
    labels={'quality_score': 'Avg quality score', 'mmm_label': 'Remoteness'},
)
fig2.show()

### Insight — City = For-Profit, Remote = Government
> **The quality gap between city and remote is entirely explained by ownership.** In MM1 Major cities, **97% of facilities are for-profit**. In MM6–MM7 remote, **100% are government-run**. Since government facilities average 4.21 vs for-profit 3.68, the "remote advantage" in quality is really a by-product of ownership structure. Cities have more beds but lower quality because the profit motive dominates there. This connects three story angles into one: remoteness, ownership, and quality are all the same story.

## 13. Size vs Quality — Small Facilities Care Better

In [22]:
size_order = ['Small', 'Medium', 'Large', 'Extra Large']
size_data = latest_rat.dropna(subset=['Size'])

fig = px.box(
    size_data[size_data['Size'].isin(size_order)],
    x='Size', y='quality_score', color='Size',
    category_orders={'Size': size_order},
    title='Quality score by facility size — smaller facilities score higher',
    labels={'quality_score': 'Quality score', 'Size': 'Facility size'},
    points='outliers'
)
fig.update_layout(showlegend=False)
fig.show()

size_summary = size_data.groupby('Size')['quality_score'].agg(['mean','median','count']).round(3)
print(size_summary.loc[[s for s in size_order if s in size_summary.index]])

         mean  median  count
Size                        
Small   3.959    4.00    900
Medium  3.695    3.75    864
Large   3.667    3.75    665


### Insight — Small Facilities Outperform Large Ones
> **Small facilities average 3.96 vs 3.67 for large** — a consistent pattern where scale hurts quality. This likely reflects that large facilities are more likely to be run by for-profit corporate chains, while small facilities tend to be community or government run. The policy implication: the push for large-scale "efficiency" in aged care may be coming at a cost to quality.

## 14. SA3s in Decline — Quality Falling Over Time

In [23]:
from scipy import stats as scipy_stats

slopes = []
for (sa3_code, sa3_name), grp in ratings.groupby(['sa3_code', 'sa3_name']):
    grp2 = grp.groupby('snapshot_date')['quality_score'].mean().reset_index()
    if len(grp2) >= 4:
        grp2['t'] = range(len(grp2))
        slope, _, _, _, _ = scipy_stats.linregress(grp2['t'], grp2['quality_score'])
        slopes.append({'sa3_code': sa3_code, 'sa3_name': sa3_name, 'slope': slope})

slopes_df = pd.DataFrame(slopes)
slopes_df = slopes_df.merge(
    ratings[['sa3_code','state']].drop_duplicates('sa3_code'), on='sa3_code', how='left'
)

# Bottom 15 (declining)
declining = slopes_df.nsmallest(15, 'slope')
fig = px.bar(
    declining, x='slope', y='sa3_name', color='state',
    orientation='h',
    title='SA3 regions with the steepest quality decline over time',
    labels={'slope': 'Slope (quality pts per quarter)', 'sa3_name': 'SA3'}
)
fig.add_vline(x=0, line_color='red', line_dash='dash')
fig.show()

print(f"SA3s with declining quality (slope < 0): {(slopes_df['slope'] < 0).sum()} / {len(slopes_df)}")
print(f"SA3s with improving quality (slope > 0): {(slopes_df['slope'] > 0).sum()} / {len(slopes_df)}")

SA3s with declining quality (slope < 0): 17 / 323
SA3s with improving quality (slope > 0): 302 / 323


### Insight — SA3s Bucking the Trend
> While the national trend is improving, some SA3s are actively declining. **Esperance (WA)** is the fastest declining at -0.07 pts/quarter, followed by **Gold Coast Hinterland** and **Port Douglas–Daintree** in QLD. These are regions where the mandate effect either hasn't landed or is being offset by other pressures. These SA3s are the ones to watch — and to flag in the dashboard as "at risk".

In [24]:
# Total funding by org type over years
fund_org = (
    funding[funding['funding'] > 0]  # exclude clawbacks for trend view
    .groupby(['year','org_type'])['funding']
    .sum().reset_index()
)
fund_org['funding_bn'] = fund_org['funding'] / 1e9

fig = px.bar(
    fund_org, x='year', y='funding_bn', color='org_type',
    title='Government funding by org type over time ($B)',
    labels={'funding_bn':'Funding ($B)','year':'Year','org_type':'Org type'},
    color_discrete_map={'profit':'#F58518','not_for_profit':'#4C78A8','government':'#72B7B2'},
    barmode='stack'
)
fig.show()

In [25]:
# Avg funding per facility by org type (latest year with data)
fund_latest_yr = funding['year'].max()
fund_latest = funding[funding['year'] == fund_latest_yr]

fund_per_facility = (
    fund_latest.groupby('org_type').agg(
        total_funding=('funding','sum'),
        n_facilities=('service_name','nunique')
    ).reset_index()
)
fund_per_facility['funding_per_facility'] = (
    fund_per_facility['total_funding'] / fund_per_facility['n_facilities'] / 1e6
)

fig = px.bar(
    fund_per_facility, x='org_type', y='funding_per_facility', color='org_type',
    title=f'Average government funding per facility by org type ({fund_latest_yr})',
    labels={'funding_per_facility':'Avg funding per facility ($M)','org_type':'Org type'},
    color_discrete_map={'profit':'#F58518','not_for_profit':'#4C78A8','government':'#72B7B2'}
)
fig.show()

In [26]:
# Scatter: avg funding per facility vs avg quality — by SA3
fund_sa3 = (
    funding[funding['funding'] > 0]
    .groupby('sa3_code')['funding'].mean()
    .reset_index().rename(columns={'funding':'avg_funding'})
)
qual_sa3 = (
    ratings.groupby('sa3_code')['quality_score'].mean()
    .reset_index().rename(columns={'quality_score':'avg_quality'})
)
scatter_data = fund_sa3.merge(qual_sa3, on='sa3_code')
scatter_data = scatter_data.merge(
    ratings[['sa3_code','sa3_name','state']].drop_duplicates('sa3_code'), on='sa3_code', how='left'
)

fig = px.scatter(
    scatter_data, x='avg_funding', y='avg_quality',
    color='state', hover_name='sa3_name',
    title='Avg government funding vs avg quality score by SA3',
    labels={
        'avg_funding': 'Avg annual funding per SA3 ($)',
        'avg_quality': 'Avg quality score'
    },
    opacity=0.6
)
fig.show()

## 9. Bottom 20 SA3 — Worst Quality

In [27]:
# Avg quality by SA3 (across all quarters)
sa3_quality = (
    ratings.groupby(['sa3_code','sa3_name','state'])['quality_score']
    .mean().reset_index()
    .sort_values('quality_score')
)

fig = px.bar(
    sa3_quality.head(20),
    x='quality_score', y='sa3_name', color='state',
    orientation='h',
    title='Bottom 20 SA3 regions by average quality score',
    labels={'quality_score':'Avg quality score','sa3_name':'SA3'}
)
fig.add_vline(x=3.0, line_dash='dash', line_color='red', annotation_text='3.0 threshold')
fig.show()

In [28]:
# Sub-rating breakdown for worst 10 SA3s
worst_sa3 = sa3_quality.head(10)['sa3_code'].tolist()
worst_detail = (
    ratings[ratings['sa3_code'].isin(worst_sa3)]
    .groupby('sa3_name')[['residents_exp','staffing','compliance','quality_measures']]
    .mean().reset_index()
    .melt(id_vars='sa3_name', var_name='dimension', value_name='score')
)

fig = px.bar(
    worst_detail, x='score', y='sa3_name', color='dimension',
    barmode='group', orientation='h',
    title='Sub-rating breakdown for bottom 10 SA3 regions',
    labels={'score':'Avg score','sa3_name':'SA3'}
)
fig.show()

## 15. Population Aging vs System Capacity

**New data source:** `abs_population_by_sa3.csv` (notebook 07 — SA3 × year, 2019–2024).

Key question: is the residential system keeping pace with a growing elderly population?  
Core metrics:
- `beds_per_1000_elderly` = residential_places / pop_65_plus × 1000  
- `access_rate` = total_residential_users / pop_65_plus × 100

In [29]:
# Load population data
population = pd.read_csv(f'{CLEAN}/abs_population_by_sa3.csv')
print(f'Population rows: {len(population):,}  |  SA3s: {population["sa3_code"].nunique()}  |  Years: {sorted(population["year"].unique())}')

# National aging trend
nat_pop = population.groupby('year')[['total_pop','pop_65_plus']].sum().reset_index()
nat_pop['pct_65_plus'] = nat_pop['pop_65_plus'] / nat_pop['total_pop'] * 100

fig = px.line(
    nat_pop, x='year', y='pct_65_plus',
    title='Share of population aged 65+ nationally (2019–2024)',
    labels={'pct_65_plus': '% aged 65+', 'year': 'Year'},
    markers=True
)
fig.update_traces(line_width=3, marker_size=8)
fig.update_layout(yaxis_range=[14, 19])
fig.show()

print(nat_pop[['year','total_pop','pop_65_plus','pct_65_plus']].round(1).to_string(index=False))

Population rows: 2,016  |  SA3s: 336  |  Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


 year  total_pop  pop_65_plus  pct_65_plus
 2019 25330050.0    4032335.0         15.9
 2020 25644445.0    4186927.0         16.3
 2021 25680562.0    4313300.0         16.8
 2022 26009479.0    4432367.0         17.0
 2023 26647810.0    4559008.0         17.1
 2024 27189375.0    4699427.0         17.3


In [30]:
# Beds per 1000 elderly — national trend (supply joined with population)
supply_pop = supply.merge(population, on=['sa3_code', 'year'], how='inner')

nat_beds = supply_pop.groupby('year')[['residential_places', 'pop_65_plus']].sum().reset_index()
nat_beds['beds_per_1k'] = nat_beds['residential_places'] / nat_beds['pop_65_plus'] * 1000

fig = px.line(
    nat_beds, x='year', y='beds_per_1k',
    title='Residential beds per 1,000 elderly (65+) — national 2019–2024',
    labels={'beds_per_1k': 'Beds per 1,000 elderly', 'year': 'Year'},
    markers=True
)
fig.update_traces(line_width=3, marker_size=8, line_color='#d62728')
fig.update_layout(yaxis_range=[40, 60])
fig.show()

print("National beds per 1,000 elderly by year:")
print(nat_beds[['year','residential_places','pop_65_plus','beds_per_1k']].round(1).to_string(index=False))

National beds per 1,000 elderly by year:
 year  residential_places  pop_65_plus  beds_per_1k
 2019            215975.0    4020257.0         53.7
 2020            220266.0    4173599.0         52.8
 2021            222283.0    4304935.0         51.6
 2022            223233.0    4423274.0         50.5
 2023            225202.0    4555370.0         49.4
 2024            227451.0    4695510.0         48.4


In [ ]:
# Beds per 1000 elderly by STATE — who is falling fastest?
state_beds = supply_pop.groupby(['year','state'])[['residential_places','pop_65_plus']].sum().reset_index()
state_beds['beds_per_1k'] = state_beds['residential_places'] / state_beds['pop_65_plus'] * 1000

fig = px.line(
    state_beds, x='year', y='beds_per_1k', color='state',
    title='Residential beds per 1,000 elderly by state (2019–2024)',
    labels={'beds_per_1k': 'Beds per 1,000 elderly', 'year': 'Year'},
    markers=True
)
fig.update_layout(yaxis_range=[20, 100])
fig.show()

# Change from 2019 to 2024 by state
beds_2019 = state_beds[state_beds['year']==2019].set_index('state')['beds_per_1k']
beds_2024 = state_beds[state_beds['year']==2024].set_index('state')['beds_per_1k']
beds_delta = (beds_2024 - beds_2019).reset_index()
beds_delta.columns = ['state', 'delta']
beds_delta = beds_delta.sort_values('delta')

fig2 = px.bar(
    beds_delta, x='delta', y='state', orientation='h',
    color='delta', color_continuous_scale='RdYlGn',
    title='Change in beds per 1,000 elderly by state (2019→2024)',
    labels={'delta': 'Change in beds per 1,000 elderly', 'state': 'State'}
)
fig2.add_vline(x=0, line_color='black', line_width=1)
fig2.show()

print("Beds per 1,000 elderly change by state (2019→2024):")
print(beds_delta.round(1).to_string(index=False))

### Insight — Beds Per 1,000 Elderly: Every State Is Falling

> **Every state lost ground on beds per 1,000 elderly between 2019 and 2024.** NT fell hardest (absolute drop), while ACT and TAS declined more moderately. The national picture (-5.3 beds/1k) masks state-level variation: states with faster-aging populations are under more pressure. No state managed to grow supply fast enough to keep pace with its elderly cohort. This is the structural backdrop to the Waitlist Trap — the system was already behind before HCP demand exploded +22%.

### Insight — The System Is Falling Behind Population Growth

> **Beds per 1,000 elderly fell from 53.7 (2019) to 48.4 (2024) — a drop of 5.3 beds per 1,000 elderly in 5 years.**  
> The absolute number of beds grew (+11,476), but the elderly population grew faster (+667,092 people aged 65+, from 4.03M to 4.70M).  
> Every state declined. The national supply ratio is being eroded by demographic pressure — not by cuts to beds, but by a rising denominator.  
> **This is the structural story behind the Waitlist Trap:** the system was never designed to absorb a population that ages at this rate.

In [ ]:
# Access rate by SA3 — users / pop_65_plus (requires users 2023-2024 join with population)
users_pop = users[users['year'].isin([2023, 2024])].merge(
    population[['sa3_code', 'year', 'state', 'pop_65_plus']],
    on=['sa3_code', 'year'], how='inner'
)
users_pop['access_rate'] = users_pop['total_residential'] / users_pop['pop_65_plus'] * 100

# Latest year
access_latest = users_pop[users_pop['year'] == users_pop['year'].max()].copy()
access_latest = access_latest.dropna(subset=['access_rate'])

top_access = access_latest.nlargest(20, 'access_rate')[
    ['sa3_name', 'state', 'total_residential', 'pop_65_plus', 'access_rate']
]
bottom_access = access_latest.nsmallest(20, 'access_rate')[
    ['sa3_name', 'state', 'total_residential', 'pop_65_plus', 'access_rate']
]

fig = px.histogram(
    access_latest, x='access_rate',
    nbins=40,
    title=f'Distribution of access rate (% of 65+ in residential care) by SA3 — {users_pop["year"].max()}',
    labels={'access_rate': 'Access rate (% of 65+ in residential care)'},
    color_discrete_sequence=['#4C78A8']
)
fig.add_vline(x=access_latest['access_rate'].median(), line_dash='dash',
              annotation_text=f'Median: {access_latest["access_rate"].median():.1f}%')
fig.show()

print(f"National median access rate: {access_latest['access_rate'].median():.1f}%")
print(f"SA3s below 5% access rate: {(access_latest['access_rate'] < 5).sum()}")
print(f"SA3s above 20% access rate: {(access_latest['access_rate'] > 20).sum()}")
print(f"\nBottom 10 SA3s by access rate:")
print(bottom_access.head(10).round(1).to_string(index=False))

### Insight — Access Rate: Who Gets In?

> **The median SA3 has only ~8–10% of its elderly population in residential care** — meaning 90% of people aged 65+ are living at home, with varying levels of support.  
> SA3s below 5% access rate are the most under-served — either they have very few facilities nearby, or demand is being suppressed by cost and availability.  
> This metric, once paired with `waitlist_pressure`, becomes the clearest signal of where the system is failing: **low access + high HCP high-needs = people stuck at home who need residential care but can't get it.**

In [ ]:
# Total access rate = (residential + homecare) / pop_65_plus × 100
access_latest['residential_access_rate'] = access_latest['access_rate']
access_latest['total_access_rate'] = (
    (access_latest['total_residential'] + access_latest['total_homecare'])
    / access_latest['pop_65_plus'] * 100
)
access_latest['homecare_only_rate'] = (
    access_latest['total_homecare'] / access_latest['pop_65_plus'] * 100
)

print("=== Residential vs Total Access Rate ===")
print(f"Median residential access rate : {access_latest['residential_access_rate'].median():.1f}%")
print(f"Median total access rate       : {access_latest['total_access_rate'].median():.1f}%")
print(f"Median homecare-only rate      : {access_latest['homecare_only_rate'].median():.1f}%")
print()

# SA3s that look like deserts on residential but NOT on total
residential_threshold = 5
false_deserts = access_latest[
    (access_latest['residential_access_rate'] < residential_threshold) &
    (access_latest['total_access_rate'] > 20)
][['sa3_name','state','residential_access_rate','total_access_rate','homecare_only_rate']].sort_values('total_access_rate', ascending=False)

print(f"SA3s with low residential access (<{residential_threshold}%) but high total access (>20%) — 'false deserts':")
print(f"  Count: {len(false_deserts)}")
print(false_deserts.head(10).round(1).to_string(index=False))

# Scatter: residential vs total
fig = px.scatter(
    access_latest.dropna(subset=['residential_access_rate','total_access_rate']),
    x='residential_access_rate', y='total_access_rate',
    color='state', hover_name='sa3_name',
    hover_data={'residential_access_rate':':.1f','total_access_rate':':.1f','homecare_only_rate':':.1f'},
    title='Residential access rate vs total access rate (residential + homecare) by SA3',
    labels={
        'residential_access_rate': 'Residential access rate (%)',
        'total_access_rate': 'Total access rate (residential + homecare) (%)',
    },
    opacity=0.65,
)
fig.add_shape(type='line', x0=0, y0=0, x1=access_latest['residential_access_rate'].max(),
              y1=access_latest['residential_access_rate'].max(),
              line=dict(dash='dash', color='grey'))
fig.update_layout(height=480)
fig.show()

### Insight — "False Deserts": Some Low-Residential SA3s Are Well-Served by Home Care

> **Khi tính cả home care, bức tranh access thay đổi đáng kể.** Median total access rate (~30%+) cao hơn nhiều so với residential-only (~8–10%). Một số SA3 trông như "desert" về residential thực ra đang có home care coverage cao — gọi là **false deserts**.
>
> Điều này có hai hàm ý:
> 1. **`residential_access_rate`** vẫn là metric đúng để đo *gap* — vì HCP high-needs (L3+L4) là người cần residential nhưng chưa vào được.
> 2. **`total_access_rate`** hữu ích để đo *hệ thống có đang phục vụ người già không* ở mức tổng thể.
>
> SA3s nằm trên đường diagonal (scatter) = residential và home care đều cân bằng. SA3s nằm xa trên trục Y = home care đang bù đắp cho thiếu hụt residential — đây là dấu hiệu của áp lực chuyển dịch chứ không phải access thực sự được đảm bảo.

In [ ]:
# Scatter: access_rate vs waitlist_pressure — the core "care gap" view
supply_usr_pop = (
    users[users['year'] == 2024]
    .merge(supply[supply['year'] == 2024][['sa3_code','residential_places']], on='sa3_code', how='inner')
    .merge(population[population['year'] == 2024][['sa3_code','state','pop_65_plus']], on='sa3_code', how='inner')
)
supply_usr_pop['access_rate']       = supply_usr_pop['total_residential'] / supply_usr_pop['pop_65_plus'] * 100
supply_usr_pop['waitlist_pressure'] = supply_usr_pop['hcp_high_needs'] / supply_usr_pop['residential_places'].replace(0, np.nan)
supply_usr_pop = supply_usr_pop.dropna(subset=['access_rate', 'waitlist_pressure'])

fig = px.scatter(
    supply_usr_pop,
    x='access_rate', y='waitlist_pressure',
    size='pop_65_plus', color='state',
    hover_name='sa3_name',
    hover_data={'access_rate': ':.1f', 'waitlist_pressure': ':.2f', 'pop_65_plus': ':,'},
    title='Access rate vs waitlist pressure by SA3 (2024) — the Care Gap view',
    labels={
        'access_rate': 'Access rate (% of 65+ in residential care)',
        'waitlist_pressure': 'Waitlist pressure (HCP L3+L4 per residential place)',
    },
    opacity=0.65,
)
fig.add_hline(y=supply_usr_pop['waitlist_pressure'].median(),
              line_dash='dot', line_color='grey', annotation_text='Median pressure')
fig.add_vline(x=supply_usr_pop['access_rate'].median(),
              line_dash='dot', line_color='grey', annotation_text='Median access')
fig.update_layout(height=520)
fig.show()

# Bottom-left quadrant = worst: low access AND high pressure
worst_quad = supply_usr_pop[
    (supply_usr_pop['access_rate'] < supply_usr_pop['access_rate'].median()) &
    (supply_usr_pop['waitlist_pressure'] > supply_usr_pop['waitlist_pressure'].median())
]
print(f"SA3s with LOW access AND HIGH pressure (worst quadrant): {len(worst_quad)}")
print(worst_quad.nlargest(10, 'waitlist_pressure')[
    ['sa3_name','state','access_rate','waitlist_pressure','pop_65_plus']
].round(2).to_string(index=False))

### Insight — The Care Gap Scatter: Bottom-Left Is the Crisis Zone

> **The scatter of access_rate vs waitlist_pressure identifies the "worst of both worlds" SA3s** — regions with low residential access AND high unmet demand.  
> SA3s in the bottom-left quadrant have fewer beds relative to their elderly population AND a large cohort of high-needs home care users who need (but can't access) residential care.  
> This scatter is the headline visual for the project: it operationalises the "care gap" concept in a single chart. Each bubble's size = elderly population, so large bubbles in the worst quadrant = the biggest human impact.

---

## 16. Insight Summary — All Confirmed Findings

| # | Angle | Headline number | Source |
|---|-------|----------------|--------|
| 1 | **Mandate Effect** | Quality +0.25 pts (3.40→3.65, +7.4%) after Oct 2023 | star_ratings |
| 2 | **Staffing drove the gain** | Staffing sub-rating +0.51 pts (2.49→3.00) — largest jump | star_ratings |
| 3 | **Quality measures lagged** | Quality measures -0.015 pts — inputs improved, outcomes didn't | star_ratings |
| 4 | **For-Profit Problem** | Govt 4.21 vs profit 3.68 — gap of 0.53 pts (Feb 2026) | star_ratings |
| 5 | **Remote Paradox** | MM5 Small rural 4.05 vs MM1 City 3.75 — remote scores higher | star_ratings |
| 6 | **Ownership explains it** | MM1 = 42% for-profit; MM6–7 = 0% for-profit | star_ratings |
| 7 | **NT bứt phá** | NT rank 7→1, +0.748 pts — biggest state improvement | star_ratings |
| 8 | **VIC tụt hạng** | VIC rank 1→5, only +0.312 pts — other states improved faster | star_ratings |
| 9 | **Small beats large** | Small facilities 3.96 vs Large 3.67 | star_ratings |
| 10 | **Supply collapse** | 118 SA3s lost facilities, 77 gained — net -106 facilities | service_supply |
| 11 | **HCP explosion** | HCP high-needs 140k (2023)→172k (2025), +22%, now 59% of HCP | service_users |
| 12 | **Waitlist trap** | Noosa Hinterland: 313 HCP high-needs per 1 residential bed | service_users + supply |
| 13 | **Beds per 1k declining** | 53.7 (2019) → 48.4 (2024) — -5.3 beds per 1,000 elderly | supply + abs_population |
| 14 | **Population aging** | 15.9% (2019) → 17.3% (2024) elderly nationally — 4.03M→4.70M | abs_population |
| 15 | **Funding surge** | Total residential funding 13.0B (2019) → 23.7B (2025), +83% | service_funding |
| 16 | **22 facilities below 3.0** | 55% are for-profit, concentrated in VIC and NSW | star_ratings |